# 08 — Limpieza final: filtros anti-bot, no-EEUU y dominancia de usuarios

Este notebook **no toca la API**. Carga el `scam_us_CONSOLIDATED_clean.csv` que generaste en la tirada consolidada y aplica tres filtros adicionales detectados en la auditoría:

## Problemas detectados

| Problema | Tweets afectados | % corpus |
|---|---|---|
| Ubicaciones claramente no-EEUU (Goa, India, Canada, UK, Nigeria...) | ~243 | 10.5% |
| 8 usuarios concentran ~10% del corpus (sesgo de bots/cuentas-news) | 235 | 10.1% |
| Usernames con patrones de bot (`hub`, `news`, `bot`, `alert`...) | 147 | 6.3% |

## Tres ajustes aplicados

1. **Filtro de ubicaciones no-EEUU** — descarta tweets cuya `user_location` contenga palabras que indican origen extranjero (Goa, India, Canada, UK, Nigeria, Pakistan, etc.).
2. **Filtro de bots por nombre** — descarta cuentas cuyo username termina o contiene patrones típicos de bots y media (`hub`, `news`, `bot`, `alert`, `tracer`, `auto`).
3. **Cap por usuario** — máximo **10 tweets por usuario** (no más). Si una cuenta tiene 81 tweets, solo conservamos 10 (los primeros cronológicamente). Esto previene que BERTopic detecte el estilo de un solo bot como tópico.

## Resultado esperado

- Partida: 2.324 tweets.
- Tras filtros: ~1.900 tweets.
- Distribución por categoría sigue equilibrada.

Coste API: **0 llamadas**.


In [1]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

# Cargar el corpus consolidado
df = pd.read_csv("../data/raw/scam_us_CONSOLIDATED_clean.csv")
print(f"Tweets cargados: {len(df)}")
print(f"Columnas: {len(df.columns)}")


Tweets cargados: 2324
Columnas: 40


## Ajuste 1 — Filtro de ubicaciones claramente no-EEUU

Lista construida a partir de la auditoría real del corpus. Es **inclusiva** (lista de exclusión explícita), no exclusiva — esto evita matar a Indiana por ambigüedad.


In [2]:
# Patrones que indican ubicación NO-EEUU
# Usamos palabras completas con \b para evitar falsos positivos
NON_US_LOCATIONS = re.compile(
    r"\b("
    # India
    r"india|goa|panaji|mumbai|delhi|bangalore|chennai|kolkata|hyderabad|"
    r"pune|ahmedabad|jaipur|surat|lucknow|"
    # Canada
    r"canada|toronto|montreal|vancouver|ottawa|calgary|edmonton|"
    r"quebec|ontario|alberta|manitoba|saskatchewan|"
    # UK / Europe
    r"united kingdom|uk\b|london|manchester|liverpool|birmingham|"
    r"edinburgh|glasgow|england|scotland|wales|ireland|dublin|"
    r"france|paris|germany|berlin|spain|madrid|italy|rome|"
    r"netherlands|amsterdam|belgium|sweden|norway|denmark|finland|"
    # Asia (excluyendo EEUU obviamente)
    r"pakistan|karachi|islamabad|lahore|"
    r"philippines|manila|"
    r"china|beijing|shanghai|hong kong|"
    r"japan|tokyo|"
    r"singapore|malaysia|thailand|bangkok|vietnam|hanoi|"
    r"south korea|seoul|"
    r"indonesia|jakarta|"
    # África
    r"nigeria|lagos|abuja|ilorin|"
    r"south africa|johannesburg|cape town|"
    r"kenya|nairobi|"
    r"egypt|cairo|"
    # América Latina
    r"mexico|mexico city|guadalajara|"
    r"brazil|sao paulo|rio|brasilia|"
    r"argentina|buenos aires|"
    r"colombia|bogota|"
    r"venezuela|caracas|"
    r"chile|santiago|"
    r"peru|lima|"
    # Oceanía
    r"australia|sydney|melbourne|brisbane|"
    r"new zealand|auckland|"
    # Otros
    r"russia|moscow|"
    r"turkey|istanbul|"
    r"israel|tel aviv|"
    r"dubai|uae|qatar|saudi arabia"
    r")\b",
    re.IGNORECASE,
)

# OJO: para evitar falsos positivos con estados/ciudades de EEUU que se llaman
# parecido (p.ej. "London, KY" hay en EEUU), comprobamos primero si la location
# tiene una indicación EXPLÍCITA de estado USA y, si la tiene, no la marcamos como non-USA.
US_STATE_INDICATORS = re.compile(
    r"\b("
    r"alabama|alaska|arizona|arkansas|california|colorado|connecticut|"
    r"delaware|florida|georgia|hawaii|idaho|illinois|indiana|iowa|"
    r"kansas|kentucky|louisiana|maine|maryland|massachusetts|michigan|"
    r"minnesota|mississippi|missouri|montana|nebraska|nevada|"
    r"new hampshire|new jersey|new mexico|new york|north carolina|"
    r"north dakota|ohio|oklahoma|oregon|pennsylvania|rhode island|"
    r"south carolina|south dakota|tennessee|texas|utah|vermont|"
    r"virginia|washington|west virginia|wisconsin|wyoming|"
    r"usa|united states|u\.s\.|u\.s\.a|america|"
    # Abreviaturas de estados (con coma o espacio antes)
    r",\s*(al|ak|az|ar|ca|co|ct|de|fl|ga|hi|id|il|in|ia|ks|ky|la|"
    r"me|md|ma|mi|mn|ms|mo|mt|ne|nv|nh|nj|nm|ny|nc|nd|oh|ok|or|pa|"
    r"ri|sc|sd|tn|tx|ut|vt|va|wa|wv|wi|wy)\b"
    r")\b",
    re.IGNORECASE,
)

def is_clearly_non_us(location):
    """True si la ubicación menciona explícitamente un sitio NO-EEUU
    y NO menciona un estado/USA explícito."""
    if not isinstance(location, str) or not location.strip():
        return False
    loc = location.lower()
    if NON_US_LOCATIONS.search(loc):
        # Si también menciona un estado USA explícito, asumimos que es residente USA
        # con vínculos al otro país (ej: "Toronto / Los Angeles" → quedarse)
        if US_STATE_INDICATORS.search(loc):
            return False
        return True
    return False

# Test rápido
test_locs = [
    ("Panaji,Goa", True),
    ("Toronto, Canada", True),
    ("Indiana", False),                # estado de EEUU
    ("Indiana, USA", False),
    ("Indianapolis, IN", False),
    ("London and New York", False),    # mixto
    ("Karachi, Pakistan", True),
    ("United States", False),
    ("USA", False),
    ("Southern California", False),
    ("New York, NY", False),
    ("nan", False),
    ("", False),
    ("CANADA 🇨🇦", True),
    ("Thames Valley, UK", True),
]
print("=== TEST DEL FILTRO DE UBICACIÓN ===")
for loc, expected in test_locs:
    got = is_clearly_non_us(loc)
    status = "✅" if got == expected else "❌"
    print(f"  {status}  expected={str(expected):5}  got={str(got):5}  | {loc!r}")


=== TEST DEL FILTRO DE UBICACIÓN ===
  ✅  expected=True   got=True   | 'Panaji,Goa'
  ✅  expected=True   got=True   | 'Toronto, Canada'
  ✅  expected=False  got=False  | 'Indiana'
  ✅  expected=False  got=False  | 'Indiana, USA'
  ✅  expected=False  got=False  | 'Indianapolis, IN'
  ✅  expected=False  got=False  | 'London and New York'
  ✅  expected=True   got=True   | 'Karachi, Pakistan'
  ✅  expected=False  got=False  | 'United States'
  ✅  expected=False  got=False  | 'USA'
  ✅  expected=False  got=False  | 'Southern California'
  ✅  expected=False  got=False  | 'New York, NY'
  ✅  expected=False  got=False  | 'nan'
  ✅  expected=False  got=False  | ''
  ✅  expected=True   got=True   | 'CANADA 🇨🇦'
  ✅  expected=True   got=True   | 'Thames Valley, UK'


## Ajuste 2 — Filtro de usernames sospechosos de bot/news

Patrón refinado para cazar bots y cuentas de news regionales sin matar usernames legítimos.


In [3]:
BOT_NEWS_USERNAME = re.compile(
    r"("
    r"_bot$|"            # termina en _bot (ej: SmishAlertBot)
    r"bot\d*$|"         # termina en bot o bot+números
    r"^bot_|"            # empieza con bot_
    r"newshub$|"         # ej: goanewshub
    r"news$|"            # cuentas que terminan en news
    r"^news_|"           # ej: news_alert
    r"alerts?$|"         # SmishAlert, ScamAlerts
    r"^alert_|"
    r"tracer$|"
    r"^auto_|"           # autobot, autopost
    r"spam$|"
    r"_hub$|"            # datalog_hub, finance_hub
    r"hubs?$"            # datalog_hubs, infohubs
    r")",
    re.IGNORECASE,
)

# Whitelist explícita por si caemos en falso positivo (cuentas legítimas con esos sufijos)
USERNAME_WHITELIST = set()  # vacía por ahora; añadir si vemos algún falso positivo

def is_suspected_bot_username(username):
    if not isinstance(username, str) or not username.strip():
        return False
    if username.lower() in USERNAME_WHITELIST:
        return False
    return bool(BOT_NEWS_USERNAME.search(username))

# Test
test_usernames = [
    ("goanewshub", True),
    ("CryptoGoblinBot", True),
    ("SmishAlert", True),
    ("CyberScoopNews", True),
    ("datalog_hubs", True),
    ("MarkADyson", False),
    ("Go511", False),
    ("Joveira132", False),
    ("realDonaldTrump", False),
    ("echo369ss", False),
]
print("=== TEST DE BOT-USERNAME ===")
for user, expected in test_usernames:
    got = is_suspected_bot_username(user)
    status = "✅" if got == expected else "❌"
    print(f"  {status}  expected={str(expected):5}  got={str(got):5}  | @{user}")


=== TEST DE BOT-USERNAME ===
  ✅  expected=True   got=True   | @goanewshub
  ✅  expected=True   got=True   | @CryptoGoblinBot
  ✅  expected=True   got=True   | @SmishAlert
  ✅  expected=True   got=True   | @CyberScoopNews
  ✅  expected=True   got=True   | @datalog_hubs
  ✅  expected=False  got=False  | @MarkADyson
  ✅  expected=False  got=False  | @Go511
  ✅  expected=False  got=False  | @Joveira132
  ✅  expected=False  got=False  | @realDonaldTrump
  ✅  expected=False  got=False  | @echo369ss


## Ajuste 3 — Cap por usuario

Limita a **máximo 10 tweets por usuario**. Si una cuenta tiene 81 publicaciones, nos quedamos solo con 10. Esto previene que BERTopic detecte el "estilo de redacción" de un único usuario como un tópico artificial.

10 es un compromiso razonable: deja suficiente para que las cuentas activas sigan representadas, pero corta la dominancia.


In [4]:
MAX_TWEETS_PER_USER = 10

# Test mental: si la distribución original era
#   goanewshub: 81 → quedarían 10
#   Go511: 53 → quedarían 10
#   MarkADyson: 33 → quedarían 10
#   Joveira132: 17 → quedarían 10
#   CryptoGoblinBot: 16 → quedarían 10
#   datalog_hubs: 14 → quedarían 10
#   GeorgeCrypul: 11 → quedarían 10
#   XYOPepe: 10 → se queda con 10
# Tweets ahorrados: 81+53+33+17+16+14+11+10 - 8*10 = 155
print(f"Cap por usuario: {MAX_TWEETS_PER_USER}")
print(f"Esto evitará que cualquier cuenta domine el corpus.")


Cap por usuario: 10
Esto evitará que cualquier cuenta domine el corpus.


## Aplicación combinada de los 3 ajustes

In [5]:
df["is_clearly_non_us"] = df["user_location"].apply(is_clearly_non_us)
df["is_bot_username"]   = df["username"].apply(is_suspected_bot_username)

print(f"Total inicial:                    {len(df)}")
print(f"Marcados como non-USA explícito:  {df['is_clearly_non_us'].sum()} ({df['is_clearly_non_us'].mean()*100:.1f}%)")
print(f"Marcados como bot-username:       {df['is_bot_username'].sum()} ({df['is_bot_username'].mean()*100:.1f}%)")
print(f"Marcados por alguno de los dos:   {(df['is_clearly_non_us'] | df['is_bot_username']).sum()}")


Total inicial:                    2324
Marcados como non-USA explícito:  245 (10.5%)
Marcados como bot-username:       160 (6.9%)
Marcados por alguno de los dos:   322


In [6]:
# Paso A: descartar non-USA y bots
df_step1 = df[~(df["is_clearly_non_us"] | df["is_bot_username"])].copy()
print(f"Tras filtros 1 y 2: {len(df_step1)}")

# Paso B: aplicar cap por usuario
# Ordenamos por created_at para quedarnos con los más recientes de cada usuario
df_step1 = df_step1.sort_values("created_at", ascending=False)
df_final = df_step1.groupby("username", group_keys=False).head(MAX_TWEETS_PER_USER).reset_index(drop=True)
print(f"Tras cap por usuario (max {MAX_TWEETS_PER_USER}/user): {len(df_final)}")

# Resumen
print()
print("=" * 60)
print("   RESUMEN LIMPIEZA v8")
print("=" * 60)
print(f"  Partida:                  {len(df)}")
print(f"  Tras filtro non-USA:      {len(df[~df['is_clearly_non_us']])}")
print(f"  Tras filtro bot-username: {len(df_step1)}")
print(f"  Tras cap por usuario:     {len(df_final)}")
print(f"  Tasa de retención:        {len(df_final)/len(df)*100:.1f}%")
print("=" * 60)

print(f"\n--- DISTRIBUCIÓN POR CATEGORÍA TRAS LIMPIEZA ---")
for cat in ['phishing', 'payment_apps', 'crypto', 'romance_monetary', 'impersonation']:
    n = df_final['query_labels'].fillna('').str.contains(cat).sum()
    pct = n / len(df_final) * 100 if len(df_final) > 0 else 0
    print(f"  {cat:<22} {n:>5}  ({pct:.1f}%)")


Tras filtros 1 y 2: 2002
Tras cap por usuario (max 10/user): 1928

   RESUMEN LIMPIEZA v8
  Partida:                  2324
  Tras filtro non-USA:      2079
  Tras filtro bot-username: 2002
  Tras cap por usuario:     1928
  Tasa de retención:        83.0%

--- DISTRIBUCIÓN POR CATEGORÍA TRAS LIMPIEZA ---
  phishing                 587  (30.4%)
  payment_apps             171  (8.9%)
  crypto                   691  (35.8%)
  romance_monetary         147  (7.6%)
  impersonation            337  (17.5%)


## Comprobaciones post-limpieza

Verificamos que ya no hay sesgos extremos por usuario ni por ubicación.


In [7]:
print("=== TOP 15 USUARIOS TRAS LIMPIEZA ===")
print(df_final["username"].value_counts().head(15))
print()
print("=== TOP 15 UBICACIONES TRAS LIMPIEZA ===")
print(df_final["user_location"].value_counts().head(15))


=== TOP 15 USUARIOS TRAS LIMPIEZA ===
username
Joveira132         10
Go511              10
XYOPepe            10
MarkADyson         10
GeorgeCrypul       10
BellizaHanhzea      9
CTRoastAgent        9
tradealgo_          8
blocktracehelp2     8
01CYBERREFUND       7
R4yt3d              7
FOX5Atlanta         6
hebrew_yisrael      6
marylynnjuszcza     6
BellsTheorem33      5
Name: count, dtype: int64

=== TOP 15 UBICACIONES TRAS LIMPIEZA ===
user_location
United States        131
USA                   70
New York, NY          35
Los Angeles, CA       27
Texas, USA            25
Chicago, IL           21
Washington, DC        21
Florida, USA          20
California, USA       20
Atlanta, GA           18
Chicago               17
San Francisco, CA     16
Austin, TX            16
New York, USA         15
New York              15
Name: count, dtype: int64


In [8]:
print("=== MUESTRA ALEATORIA DE 20 TWEETS ===\n")
for _, row in df_final.sample(min(20, len(df_final)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {str(row['text'])[:280]}")
    print()


=== MUESTRA ALEATORIA DE 20 TWEETS ===

[phishing] @ameelms — Equine Park
  Do not read your email while driving, I repeat, DO NOT.  

I almost clicked a phishing email. 

Today is not the day of me failing a phish simulation. Almost. But I did not. 💀

[crypto] @liladdies — Somewhereinmyhead
  I dunno why x be twicking mehhn, got my account temporarily locked coz I messaged someone!
@elonmusk first start by cracking down those bot randos who keeps messaging me about crypto scam, or some real-estate crap!

[romance_monetary] @ClearyJerome — West Hollywood, CA
  Ladies, if a major TV or film star starts messaging you and calling you and asking you for money....it's not the real TV or film star! Woman scammed into thinking she's dating Virgin River star Martin Henderson moves to New Zealand, loses $375K https://t.co/gBdyXoPDlT via @EW

[phishing] @FedupMama84 — Cleveland, OH
  Are we just going to forget about the fact that the FBI released those absurdly fake text messages from Charlie K

In [9]:
# 3 ejemplos por categoría
print("=== 3 EJEMPLOS POR CATEGORÍA ===\n")
for label in ['phishing', 'payment_apps', 'crypto', 'romance_monetary', 'impersonation']:
    subset = df_final[df_final["query_labels"].fillna("").str.contains(label)]
    print(f"--- {label.upper()} ({len(subset)} tweets) ---")
    for _, row in subset.sample(min(3, len(subset)), random_state=42).iterrows():
        print(f"  @{row['username']} [{row.get('user_location', '')}]")
        print(f"    {str(row['text'])[:240]}")
    print()


=== 3 EJEMPLOS POR CATEGORÍA ===

--- PHISHING (587 tweets) ---
  @honeeycrisp [Sacramento, CA]
    Everyone in the company got a phishing email but me. So now I’m jealous. Feel left out.
  @Cybersecinsider [United States]
    Updated Post: Arsen Launches Smishing Simulation to Help Companies Defend Against Mobile Phishing Threats https://t.co/eN6fqQrKho https://t.co/xBL2eHTir0
  @Alevskey [San Francisco, CA]
    Arsen Launches Smishing Simulation to Help Companies Defend Against Mobile Phishing Threats: https://t.co/owanWPjJWy by The Security Ledger with Paul F. Roberts #infosec #cybersecurity #technology #news

--- PAYMENT_APPS (171 tweets) ---
  @JET24FOX66 [Erie, Pa]
    Pennsylvania State Police are investigating after a Crawford County couple was reportedly scammed out of more than $2,000. https://t.co/y4exhILnHW
  @BabyGirl4_90 [Cali]
    Don’t fall for this “Zelle” scam! This is a fake email and no “fees” will ever be required. Stay safe ladies! #ScamAwareness https://t.co/sc9P

## Guardado del corpus final

In [10]:
import os
os.makedirs("../data/raw", exist_ok=True)

# Limpiamos columnas auxiliares antes de guardar
cols_to_drop = ["is_clearly_non_us", "is_bot_username"]
df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], inplace=True)

OUTPUT_PATH = "../data/raw/scam_us_FINAL_v8.csv"
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT_PATH}")
print(f"  {len(df_final)} tweets listos para BERTopic + clasificación zero-shot.")
print()
print("📦 Estado de archivos en data/raw/:")
print(f"   scam_us_CONSOLIDATED_clean.csv  ← bruto-limpio anterior ({len(df)})")
print(f"   scam_us_FINAL_v8.csv            ← corpus final limpio ({len(df_final)})")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del scam_us_FINAL_v8.csv en Drive/Dropbox.")
print()
print("📌 SIGUIENTE PASO: notebook 09 con BERTopic + clasificación zero-shot BART-MNLI.")


✓ Guardado: ../data/raw/scam_us_FINAL_v8.csv
  1928 tweets listos para BERTopic + clasificación zero-shot.

📦 Estado de archivos en data/raw/:
   scam_us_CONSOLIDATED_clean.csv  ← bruto-limpio anterior (2324)
   scam_us_FINAL_v8.csv            ← corpus final limpio (1928)

🚨 HAZ COPIA DE SEGURIDAD del scam_us_FINAL_v8.csv en Drive/Dropbox.

📌 SIGUIENTE PASO: notebook 09 con BERTopic + clasificación zero-shot BART-MNLI.
